# Week 05 practical: evaluating and tuning a model
**Student practical · CHS 2406: Data-driven AI · Final-year BSc · 60 minutes**

| Exercise | What you will implement | Slides in the attached lecture | Time |
|---|---|---|---|
| Setup and scenario | Regression, workflow and RMSE | **6**: Supervised Learning: Regression; **13**: Steps in Machine Learning; **16**: Evaluation | 5 min |
| **1. Hold-out split** | Separate training, validation and test data; verify 60/20/20 | **21**: Data Splitting; **22**: Hold-Out Testing | 8 min |
| **2. Underfitting and overfitting** | Fit shallow, moderate and unrestricted trees; compare training and validation errors | **18**: Model Parameters vs Hyperparameters; **20**: Model Performance | 12 min |
| **3. Manual hyperparameter tuning** | Choose tree depth using a fixed validation set | **18**: Model Parameters vs Hyperparameters; **19**: Hyperparameter Grid Search graphic; **22**: Hold-Out Testing | 8 min |
| **4. Five-fold cross-validation** | Fit fresh models on five development-data folds; summarise fold scores | **23**: 5-Fold Cross-Validation; **24**: Train–Test Split with Cross-Validation | 10 min |
| **5. Grid search** | Search 12 depth/leaf-size combinations with five-fold CV | **18–19**: hyperparameters and combinations; **23–24**: CV and refitting | 10 min |
| **6. Final hold-out test** | Evaluate the selected, refitted model once on the untouched test set | **13**: final test in the ML workflow; **22**: Hold-Out Testing; **24**: final refit and test | 7 min |
| **Optional A. Split sensitivity** | Measure how a fixed model's validation score changes across splits | **20**: Model Performance; **22–24**: hold-out and CV | +10 min |
| **Optional B. Randomised search** | Compare six sampled candidates with the full 12-candidate grid | **18–19**, **23–24**: tuning and CV | +10–15 min |
| **Optional C. Learning curves** | Examine training size, model complexity and validation error | **20**, **23–24**: fitting behaviour and CV | +15 min |

**Core total: 60 minutes.** Optional tasks are for early finishers and follow the core tasks.
Slide numbers are the 1-based PowerPoint positions in `Week05_Lecture01_Machine_Learning_Basics.pptx` (28 slides), including the title slide.

**Prepared for Dr Hamid Nasiri · University of Huddersfield.**


## Start here
In Google Colab, choose **File → Upload notebook**, select this `.ipynb` file, and use a **CPU** runtime. No Drive mount, account credentials or data download is needed. Run the setup cells, then complete each numbered exercise in order. Each `...` is a small coding task; replace it before running that cell. Hints describe the required API. Write your short explanations in the answer cells.

By the end, you should be able to explain which data train a model, which data choose its settings, and which data estimate final performance. You will also diagnose fitting behaviour from **both** training and validation errors.

### Scenario: quality control for 3D printing
A fictional manufacturing lab records **150 independent printing trials**. Each row contains nozzle temperature in **°C** and the resulting specimen tensile strength in **MPa**. Predict strength from temperature. The curved relationship and measurement noise are synthetic teaching choices, not an experimentally validated manufacturing model.

| Quantity | Role |
|---|---|
| `temperature_c` | One numerical input feature |
| `strength_mpa` | Continuous regression target |
| Learned temperature thresholds and leaf predictions | Model parameters fitted from training examples |
| `max_depth`, `min_samples_leaf` | Hyperparameters chosen before fitting |

A decision-tree regressor splits temperature into ranges and predicts the mean training strength in each final range. Increasing depth permits more splits. Requiring more examples in each leaf limits small, noisy regions.

We use **RMSE = sqrt(mean((actual − predicted)²))**. Lower is better; the unit is MPa. Squaring first penalises large errors. RMSE is a regression measure, not classification accuracy.


In [ ]:
# Colab normally includes all four packages. Install only packages that are missing.
import importlib.util
import subprocess
import sys

required = {'numpy': 'numpy', 'pandas': 'pandas',
            'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    # Equivalent to: %pip install -q numpy pandas matplotlib scikit-learn
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print('Installed:', ', '.join(missing))
else:
    print('All required packages are already available. No installation needed.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, GridSearchCV,
    RandomizedSearchCV, ParameterGrid, learning_curve
)
from sklearn.tree import DecisionTreeRegressor

SEED = 42
TRAIN_COLOUR = '#ef7d22'
VAL_COLOUR = '#1769aa'
TEST_COLOUR = '#7e43b7'
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 12)

def rmse(actual, predicted):
    # This form works across a broad range of scikit-learn versions.
    return float(np.sqrt(mean_squared_error(actual, predicted)))

def depth_label(depth):
    return 'Unlimited' if depth is None else str(depth)

def require_completed(value, name):
    if value is Ellipsis:
        raise ValueError(f'Complete the TODO for {name}: replace ... and rerun this cell.')

def show_fit(ax, model, title):
    grid_x = np.linspace(185, 235, 500).reshape(-1, 1)
    ax.scatter(X_train[:, 0], y_train, c=TRAIN_COLOUR, s=23,
               alpha=0.8, label='Training')
    ax.scatter(X_val[:, 0], y_val, c=VAL_COLOUR, marker='x',
               s=33, label='Validation')
    ax.plot(grid_x[:, 0], model.predict(grid_x), color='#202b46',
            linewidth=1.8, label='Prediction')
    ax.set(title=title, xlabel='Nozzle temperature (°C)',
           ylabel='Tensile strength (MPa)', xlim=(185, 235), ylim=(5, 48))
    ax.grid(alpha=0.18)

print('NumPy:', np.__version__, '| pandas:', pd.__version__,
      '| scikit-learn:', sklearn.__version__)


In [ ]:
# Reproducible fictional trials. Do not alter this cell during the core exercises.
rng = np.random.default_rng(SEED)
X = rng.uniform(185, 235, size=(150, 1))
mean_strength = 15 + 24 * np.exp(-((X[:, 0] - 210) / 11) ** 2)
y = mean_strength + rng.normal(0, 3, size=150)
row_ids = np.arange(len(y))
print(f'Created {len(y)} independent trials with {X.shape[1]} input feature.')
print('We will reserve the test rows before inspecting training patterns.')


## Exercise 1 · Hold-out testing · 8 minutes
**Slides 21–22.** Make one fixed **60% training / 20% validation / 20% test** split.

1. Reserve 20% of all rows as the test set. Call the remaining 80% the **development pool**.
2. Split that pool into training and validation sets. Think carefully: 20% of all data is what fraction of the remaining 80%?
3. Check the sizes and confirm that no row belongs to more than one set.

Complete the two proportions. `train_test_split(..., test_size=fraction, random_state=SEED)` returns the two parts in the order supplied below. Random splitting is suitable here because the fictional trials are independent. Time-ordered or grouped data need an appropriate split strategy.


In [ ]:
# TODO 1: replace both ... values with decimal proportions.
test_fraction = ...
validation_fraction_of_dev = ...
require_completed(test_fraction, 'test_fraction')
require_completed(validation_fraction_of_dev, 'validation_fraction_of_dev')

# Keep row identifiers alongside X and y so we can check overlap.
X_dev, X_test, y_dev, y_test, id_dev, id_test = train_test_split(
    X, y, row_ids, test_size=test_fraction, random_state=SEED
)
X_train, X_val, y_train, y_val, id_train, id_val = train_test_split(
    X_dev, y_dev, id_dev, test_size=validation_fraction_of_dev,
    random_state=SEED
)

assert (len(y_train), len(y_val), len(y_test)) == (90, 30, 30),     'Aim for 90 training, 30 validation and 30 test rows. Recheck both fractions.'
assert set(id_train).isdisjoint(id_val)
assert set(id_train).isdisjoint(id_test)
assert set(id_val).isdisjoint(id_test)
assert set(id_train) | set(id_val) | set(id_test) == set(row_ids)

split_table = pd.DataFrame({
    'Set': ['Training', 'Validation', 'Test'],
    'Rows': [len(y_train), len(y_val), len(y_test)],
    'Purpose': ['Fit parameters', 'Choose hyperparameters', 'Final evaluation only']
})
display(split_table)

fig, ax = plt.subplots(figsize=(9, 2.0))
left = 0
for name, size, colour in zip(split_table['Set'], split_table['Rows'],
                             [TRAIN_COLOUR, VAL_COLOUR, TEST_COLOUR]):
    ax.barh(0, size, left=left, color=colour, height=0.55)
    ax.text(left + size/2, 0, f'{name}\n{size} rows',
            ha='center', va='center', color='white', weight='bold')
    left += size
ax.set(xlim=(0, 150), yticks=[], xlabel='Number of rows',
       title='One fixed hold-out split')
plt.tight_layout()
plt.show()
print('Test features and targets now stay unused until Exercise 6.')


**Your explanation:** Why is the second split fraction different from the required 20% of the complete dataset? What goes wrong if you select hyperparameters using test error?

*Write 2–3 sentences here.*


## Exercise 2 · Underfitting and overfitting · 12 minutes
**Slides 18 and 20.** Keep the split fixed and vary model complexity.

Complete the list to compare a **depth-1 tree**, a **depth-4 tree**, and a tree with **unlimited depth** (`None`). Read the provided `.fit()` and `.predict()` calls. All three models learn only from the training set.

Predict before running: which model will have the lowest training error? Will that necessarily give the lowest validation error?


In [ ]:
# TODO 2: use a list containing the three requested depth settings.
depths_to_compare = ...
require_completed(depths_to_compare, 'depths_to_compare')
assert depths_to_compare == [1, 4, None], 'Use depth 1, depth 4 and unlimited depth.'

comparison_models = []
comparison_rows = []
for depth in depths_to_compare:
    model = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    model.fit(X_train, y_train)                  # Learn parameters here.
    train_predictions = model.predict(X_train)
    validation_predictions = model.predict(X_val)
    comparison_models.append(model)
    comparison_rows.append({
        'Depth limit': depth_label(depth),
        'Training RMSE': rmse(y_train, train_predictions),
        'Validation RMSE': rmse(y_val, validation_predictions),
        'Leaves': model.get_n_leaves()
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(3))
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, depth, model in zip(axes, depths_to_compare, comparison_models):
    show_fit(ax, model, f'Depth: {depth_label(depth)}')
axes[0].legend(fontsize=9)
fig.suptitle('The same training and validation rows in all three panels', y=1.03)
plt.tight_layout()
plt.show()


**Your explanation:** Identify the underfitting and overfitting models using the table **and** the plots. Which model would you currently prefer, and why? Why can near-zero training error be misleading?

*Write 3–4 sentences here.*


## Exercise 3 · Tune a hyperparameter with a fixed validation set · 8 minutes
**Slides 18–19 and 22.** Try six depth limits. We provide the training loop. Choose the row with the **smallest validation RMSE** and use its position to retrieve the depth.

`Series.idxmin()` returns the index label of the smallest value. Here the table uses the default integer index, matching the candidate list. `None` means no depth limit, not a missing experiment.


In [ ]:
candidate_depths = [1, 2, 3, 4, 6, None]
manual_rows = []
for depth in candidate_depths:
    candidate = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    candidate.fit(X_train, y_train)
    manual_rows.append({
        'Depth limit': depth_label(depth),
        'Training RMSE': rmse(y_train, candidate.predict(X_train)),
        'Validation RMSE': rmse(y_val, candidate.predict(X_val))
    })
manual_results = pd.DataFrame(manual_rows)
display(manual_results.round(3))

ax = manual_results.plot(x='Depth limit', y=['Training RMSE', 'Validation RMSE'],
                         marker='o', figsize=(8, 4),
                         color=[TRAIN_COLOUR, VAL_COLOUR])
ax.set(ylabel='RMSE (MPa)', title='Manual hold-out tuning: lower is better')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# TODO 3: find the row index with the lowest VALIDATION RMSE.
best_index = ...
require_completed(best_index, 'best_index')
assert best_index == int(manual_results['Validation RMSE'].idxmin()),     'Choose using validation RMSE, rather than training RMSE.'

selected_holdout_depth = candidate_depths[best_index]
holdout_selected_model = DecisionTreeRegressor(
    max_depth=selected_holdout_depth, random_state=SEED
).fit(X_train, y_train)
print('Selected depth:', depth_label(selected_holdout_depth))
print('Selection validation RMSE:', round(manual_results.loc[best_index, 'Validation RMSE'], 3))
print('No test score has been calculated.')


**Your explanation:** Which depth did the validation set select? Explain why a single validation score might change if we use a different random split. Which quantities in this experiment are fitted parameters, and which are hyperparameters?

*Write 2–3 sentences here.*


## Exercise 4 · Five-fold cross-validation · 10 minutes
**Slides 23–24.** We now demonstrate the alternative CV workflow on the **120-row development pool** (the original training and validation rows together). The same 30-row test set stays separate.

In each fold, fit a **fresh model** on 96 development rows and validate on 24 other development rows. Every development row is used for validation once. These are repeated model fits on different folds, not continued training of one model.

The fold diagram below is provided. Complete the `cross_validate` call for a fixed depth-4 tree. Use `X_dev`, `y_dev`, `cv=folds`, `scoring='neg_root_mean_squared_error'`, `return_train_score=True` and `n_jobs=1`. Scikit-learn maximises scores, so it represents RMSE as a negative score. **Negate it to recover ordinary positive RMSE.**


In [ ]:
folds = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_sizes = []
fig, ax = plt.subplots(figsize=(10, 3))
for fold_number, (fit_idx, validation_idx) in enumerate(folds.split(X_dev), start=1):
    ax.scatter(fit_idx, np.repeat(fold_number, len(fit_idx)),
               marker='s', s=35, color=TRAIN_COLOUR)
    ax.scatter(validation_idx, np.repeat(fold_number, len(validation_idx)),
               marker='s', s=35, color=VAL_COLOUR)
    assert set(fit_idx).isdisjoint(validation_idx)
    assert set(id_dev[fit_idx]).isdisjoint(id_test)
    assert set(id_dev[validation_idx]).isdisjoint(id_test)
    fold_sizes.append({'Fold': fold_number, 'Fit rows': len(fit_idx),
                       'Validation rows': len(validation_idx)})
ax.set(xlabel='Development-row position (not a time axis)', ylabel='Fold',
       yticks=range(1, 6), title='Orange = fit; blue = validate. Test rows are absent.')
ax.invert_yaxis()
plt.tight_layout()
plt.show()
display(pd.DataFrame(fold_sizes))


In [ ]:
# TODO 4: replace ... with the cross_validate(...) call described above.
cv_output = ...
require_completed(cv_output, 'cv_output')

cv_summary = pd.DataFrame({
    'Fold': np.arange(1, 6),
    'Training RMSE': -cv_output['train_score'],
    # In this API, test_score means the held-out CV FOLD, not our final test set.
    'Validation RMSE': -cv_output['test_score']
})
display(cv_summary.round(3))
print('Mean validation RMSE:', round(cv_summary['Validation RMSE'].mean(), 3), 'MPa')
print('Fold SD:', round(cv_summary['Validation RMSE'].std(ddof=0), 3), 'MPa')
print('This SD describes fold variation; it is not a confidence interval.')


**Your explanation:** Why are there five fitted models? Why does the API key `test_score` not mean that we have used our final test set? Can the CV score and Exercise 3's single validation score be compared as if they used identical training and validation rows?

*Write 2–3 sentences here.*


## Exercise 5 · Hyperparameter optimisation with grid search · 10 minutes
**Slides 18–19 and 23–24.** Search every combination of six depth limits and two minimum leaf sizes:

- `max_depth`: **1, 2, 3, 4, 6, None**
- `min_samples_leaf`: **1, 5**

Complete `param_grid`, then run the provided `GridSearchCV` call. The grid has **6 × 2 = 12 candidates**. With five folds, that is **60 CV fits**, followed by **one refit** of the selected settings on all 120 development rows. This implements the combination-search idea on slide 19 with tree hyperparameters.

`refit=True` means the fitted `best_estimator_` is ready for the final test. It is a new fit on the complete development pool. The best CV score is still a model-selection score because it was used to choose the winner.


In [ ]:
# TODO 5: supply a dictionary with both parameter names and candidate lists.
param_grid = ...
require_completed(param_grid, 'param_grid')
assert param_grid.get('max_depth') == [1, 2, 3, 4, 6, None]
assert param_grid.get('min_samples_leaf') == [1, 5]

combinations = list(ParameterGrid(param_grid))
print(f'{len(combinations)} candidates × {folds.n_splits} folds = '
      f'{len(combinations) * folds.n_splits} CV fits, plus one final refit.')
candidate_table = pd.DataFrame([{
    'Candidate': i + 1, 'max_depth': depth_label(p['max_depth']),
    'min_samples_leaf': p['min_samples_leaf']
} for i, p in enumerate(combinations)])
display(candidate_table)

search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=SEED),
    param_grid=param_grid, scoring='neg_root_mean_squared_error',
    cv=folds, return_train_score=True, refit=True, n_jobs=1, error_score='raise'
)
search.fit(X_dev, y_dev)           # The final test set is still excluded.

search_results = pd.DataFrame([{
    'max_depth': depth_label(params['max_depth']),
    'min_samples_leaf': params['min_samples_leaf'],
    'Mean training RMSE': -search.cv_results_['mean_train_score'][i],
    'Mean validation RMSE': -search.cv_results_['mean_test_score'][i],
    'Fold SD': search.cv_results_['std_test_score'][i],
    'Rank': int(search.cv_results_['rank_test_score'][i])
} for i, params in enumerate(search.cv_results_['params'])])
display(search_results.sort_values('Rank').round(3).reset_index(drop=True))
print('Selected hyperparameters:', search.best_params_)
print('Best mean CV RMSE:', round(-search.best_score_, 3), 'MPa')
print('Refitted on', search.best_estimator_.tree_.n_node_samples[0], 'development rows.')

heatmap = search_results.pivot(index='max_depth', columns='min_samples_leaf',
                               values='Mean validation RMSE')
heatmap = heatmap.reindex([depth_label(d) for d in param_grid['max_depth']])
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(heatmap.to_numpy(), cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(heatmap.columns)), heatmap.columns)
ax.set_yticks(range(len(heatmap.index)), heatmap.index)
ax.set(xlabel='Minimum examples per leaf', ylabel='Depth limit',
       title='Mean CV RMSE (MPa): lower is better')
for row in range(len(heatmap.index)):
    for column in range(len(heatmap.columns)):
        ax.text(column, row, f'{heatmap.iloc[row, column]:.2f}',
                ha='center', va='center', color='black',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))
fig.colorbar(image, ax=ax, label='Validation RMSE (MPa)')
plt.tight_layout()
plt.show()


**Your explanation:** Report the selected settings and the best mean CV RMSE. Did increasing minimum leaf size help the unrestricted tree? How can an unrestricted tree avoid fitting every individual training observation? How many model fits did this grid search require, including the refit?

*Write 3–4 sentences here.*


## Exercise 6 · Evaluate the final selected model · 7 minutes
**Slides 13, 22 and 24.** Freeze the grid, scoring rule and selection procedure. The final model is `search.best_estimator_`, already refitted on all development rows. Complete its prediction on `X_test`.

The provided cell also evaluates a **predefined mean-prediction baseline** fitted on the development targets. This comparison is for context and does not trigger a new model choice. Evaluate these frozen predictions once, record the result, then write your conclusion. Do not return to tuning after seeing the test score.


In [ ]:
final_model = search.best_estimator_
# TODO 6: predict strength for the untouched test features.
test_predictions = ...
require_completed(test_predictions, 'test_predictions')

baseline = DummyRegressor(strategy='mean').fit(X_dev, y_dev)
baseline_predictions = baseline.predict(X_test)
final_report = pd.DataFrame({
    'Model': ['Selected tree', 'Mean baseline (fixed in advance)'],
    'Test RMSE (MPa)': [rmse(y_test, test_predictions),
                        rmse(y_test, baseline_predictions)]
})
display(final_report.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, test_predictions, color=TEST_COLOUR, s=45)
limits = [min(y_test.min(), test_predictions.min()) - 1,
          max(y_test.max(), test_predictions.max()) + 1]
axes[0].plot(limits, limits, '--', color='gray', label='Perfect prediction')
axes[0].set(xlabel='Measured strength (MPa)', ylabel='Predicted strength (MPa)',
            title='Final test predictions', xlim=limits, ylim=limits)
axes[0].legend()
axes[1].scatter(X_test[:, 0], y_test - test_predictions, color=TEST_COLOUR, s=45)
axes[1].axhline(0, linestyle='--', color='gray')
axes[1].set(xlabel='Nozzle temperature (°C)', ylabel='Actual − predicted (MPa)',
            title='Final test residuals')
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


**Your conclusion:** State the chosen hyperparameters, mean CV RMSE and final test RMSE in MPa. Explain whether the tree improves on the predefined baseline. Why are the CV and test errors not identical? What would be wrong with expanding the grid because this test result disappointed you?

*Write 4–5 sentences here.*


## Core completion check
Before moving to the optional tasks, confirm that you can explain:

- Why the fixed split contains **90 / 30 / 30** rows.
- How both training and validation RMSE reveal underfitting or overfitting.
- Why the CV validation folds come only from the development pool.
- Why a 12-candidate, five-fold search performs **61 fits** with refitting enabled.
- Why test results do not become a new tuning signal.

**Optional-task rule:** Continue using only `X_dev` and `y_dev`. Leave the final test report unchanged. These are separate development investigations, not extra attempts to improve the already-observed test score.


## Optional A · How sensitive is one hold-out score? · 10 minutes
**Slides 20 and 22–24.** Fit a fixed depth-4 tree using **20 different train/validation splits** of the development pool. Each time, validate on 25% of that pool. Record the validation RMSE, show a histogram and report its mean, standard deviation, minimum and maximum.

**Challenge:** implement the loop using `train_test_split` and keep the model hyperparameters fixed. Do not choose the split seed with the best score. Explain how the distribution motivates using multiple folds without claiming that CV removes all evaluation uncertainty.


In [ ]:
# TODO A: create a 20-seed loop, evaluate the same depth-4 tree and plot the scores.
# Hints: for split_seed in range(20): ...
#        train_test_split(X_dev, y_dev, test_size=0.25, random_state=split_seed)
#        DecisionTreeRegressor(max_depth=4, random_state=SEED)
#        rmse(y_check, fitted_model.predict(X_check))
# Write your implementation below.
pass


**Your interpretation:** Summarise how much the score changes and why selecting the most favourable split would misrepresent performance.

*Write your answer here.*


## Optional B · Randomised search with a smaller budget · 10–15 minutes
**Slides 18–19 and 23–24.** Use `RandomizedSearchCV` with the same candidate lists, the same folds and the same scorer, but sample only **six** of the 12 combinations. Use `random_state=SEED`, `refit=True`, `n_jobs=1` and `error_score='raise'`.

Compare best **mean CV** RMSE and number of model fits with the full grid. Inspect which combinations were sampled. Explain whether a six-candidate search is guaranteed to find the grid's winner. Keep all comparisons on the development data.


In [ ]:
# TODO B: build and fit RandomizedSearchCV on X_dev and y_dev.
# Use the same param_grid, n_iter=6, cv=folds, scorer and seed.
# Access .best_params_, .best_score_ and .cv_results_['params'].
# Count fits as number_of_candidates * 5 + 1.
# Write your implementation below.
pass


**Your interpretation:** Does this run find the same winner? Explain the budget/coverage trade-off and why one successful random seed does not guarantee the same result in general.

*Write your answer here.*


## Optional C · Learning curves · 15 minutes
**Slides 20 and 23–24.** Investigate whether more training rows might help.

Use `learning_curve` on the development pool with five-fold CV, `train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0]`, `shuffle=True`, `random_state=SEED`, `scoring='neg_root_mean_squared_error'` and `n_jobs=1`. Compare an unrestricted tree (`min_samples_leaf=1`) with a fresh clone of the selected tree.

Plot mean training and validation RMSE against the actual training sizes returned by the function. Negate its returned scores before averaging. Explain that a train size of 1.0 means all rows in each fold's fitting portion, **not** all development rows and never the final test set. Discuss what you would expect if you also added a depth-1 tree.


In [ ]:
# TODO C: call learning_curve for both requested models and plot positive RMSE.
# Hints: sizes, training_scores, validation_scores = learning_curve(...)
#        mean_training_rmse = (-training_scores).mean(axis=1)
#        mean_validation_rmse = (-validation_scores).mean(axis=1)
# Use X_dev and y_dev only; create two plot panels with a shared RMSE axis.
# Write your implementation below.
pass


**Your interpretation:** Which model has the larger training–validation gap? Is the validation trend perfectly monotonic? Would more data alone necessarily fix underfitting by a depth-1 tree?

*Write your answer here.*


## API reference and further reading
The practical follows the attached lecture's evaluation workflow. These official scikit-learn pages explain the implementation details:

- [Cross-validation and holding out a final test set](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Hyperparameter search, candidate grids and refitting](https://scikit-learn.org/stable/modules/grid_search.html)
- [DecisionTreeRegressor: depth and minimum leaf size](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
- [Learning curves](https://scikit-learn.org/stable/modules/learning_curve.html)
- [Avoiding data leakage](https://scikit-learn.org/stable/common_pitfalls.html)

This tree needs no feature scaling. If you later add fitted preprocessing, fit it within each training portion, usually using a scikit-learn `Pipeline`. Keep the independent test set out of preprocessing fits and model selection.

**Save your work:** use **File → Download → Download .ipynb** in Colab. Retain your completed code, outputs and written interpretations.
